# PFA   Classification des Maladies Pulmonaires
**NIH Chest X-ray Dataset   15 Pathologies**

In [ ]:
# =============================================================
# CELLULE 1   Imports + Vérification GPU
# =============================================================
import os, glob, json, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score,
                              precision_score, recall_score, accuracy_score,
                              confusion_matrix, classification_report)
warnings.filterwarnings('ignore')

#    Chemins Kaggle   
DATA_PATH = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
WORK_PATH = '/kaggle/working'
os.makedirs(f'{WORK_PATH}/models', exist_ok=True)
os.makedirs(f'{WORK_PATH}/logs',   exist_ok=True)

IMG_SIZE   = 224
BATCH_SIZE = 32
ALL_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion',    'Emphysema',    'Fibrosis',       'Hernia',
    'Infiltration','Mass',         'No Finding',     'Nodule',
    'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'
]
N_CLASSES = len(ALL_LABELS)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print(f'GPU detecte : {gpus[0].name}')
else:
    print('Aucun GPU   active-le : Settings → Accelerator → GPU P100')

csv_path = os.path.join(DATA_PATH, 'Data_Entry_2017.csv')
print(f'Dataset NIH disponible : {os.path.exists(csv_path)}')
print(f'TensorFlow : {tf.__version__}')

In [ ]:
# =============================================================
# CELLULE 2   Chargement + Encodage + Split officiel NIH
# =============================================================

df = pd.read_csv(os.path.join(DATA_PATH, 'Data_Entry_2017.csv'))
print(f'CSV : {len(df):,} images')

all_paths = glob.glob(os.path.join(DATA_PATH, 'images_*/images/*.png'))
path_dict = {os.path.basename(p): p for p in all_paths}
df['FilePath'] = df['Image Index'].map(path_dict)
df = df.dropna(subset=['FilePath']).reset_index(drop=True)
print(f'Images indexees : {len(df):,}')

# Encodage multi-label
label_dummies = df['Finding Labels'].str.get_dummies(sep='|')
for label in ALL_LABELS:
    if label not in label_dummies.columns:
        label_dummies[label] = 0
label_dummies = label_dummies[ALL_LABELS]
df = pd.concat([df, label_dummies], axis=1)

# Split officiel
with open(os.path.join(DATA_PATH, 'train_val_list.txt')) as f:
    tv_list = set(f.read().splitlines())
with open(os.path.join(DATA_PATH, 'test_list.txt')) as f:
    te_list = set(f.read().splitlines())

df_trainval = df[df['Image Index'].isin(tv_list)].copy()
df_test     = df[df['Image Index'].isin(te_list)].copy()
df_train, df_val = train_test_split(
    df_trainval, test_size=0.2, random_state=42,
    stratify=df_trainval['No Finding']
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

print(f'Train brut : {len(df_train):,} | Val : {len(df_val):,} | Test : {len(df_test):,}')
print(f'Data leakage : {len(set(df_train["Image Index"]) & set(df_test["Image Index"]))} (doit être 0)')

In [ ]:
# =============================================================
# CELLULE 3 Visualisation avant et apres + Reequilibrage
# =============================================================

#    AVANT rééquilibrage   
counts_avant = df_train[ALL_LABELS].sum().sort_values(ascending=False)
total_avant  = len(df_train)

print('DISTRIBUTION AVANT RÉÉQUILIBRAGE')
print('='*55)
for label, count in counts_avant.items():
    ratio = count / counts_avant.max()
    print(f'{label:<25} {count:>7,}  ({count/total_avant*100:.1f}%)  ratio={ratio:.2f}')


# Reequilibrage : undersampling de No Finding

# nbre d'exemples de chaque classe (hors No Finding)
disease_counts = {
    label: df_train[df_train[label] == 1].shape[0]
    for label in ALL_LABELS if label != 'No Finding'
}

# Cible : max des classes maladies (Infiltration)
TARGET_PER_CLASS = min(max(disease_counts.values()), 4500)
NO_FINDING_LIMIT = TARGET_PER_CLASS

print(f'\nCible par classe (TARGET) : {TARGET_PER_CLASS:,}')
print(f'No Finding limité à       : {NO_FINDING_LIMIT:,} (était ~{counts_avant["No Finding"]:,})')

sampled = []
for label in ALL_LABELS:
    limit  = NO_FINDING_LIMIT if label == 'No Finding' else TARGET_PER_CLASS
    subset = df_train[df_train[label] == 1].sample(
        n=min(limit, df_train[df_train[label] == 1].shape[0]),
        random_state=42
    )
    sampled.append(subset)

df_train_s = pd.concat(sampled).drop_duplicates(subset='Image Index')
df_train_s = df_train_s.sample(frac=1, random_state=42).reset_index(drop=True)

# apres reequilibrage
counts_apres = df_train_s[ALL_LABELS].sum().sort_values(ascending=False)

print(f'\nTrain équilibré : {len(df_train_s):,} images')
print('='*55)
for label, count in counts_apres.items():
    ratio = count / counts_apres.max()
    print(f'{label:<25} {count:>7,}  ratio={ratio:.2f}')

# Visualisation avant/apres
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
fig.suptitle('Distribution du dataset   Avant vs Après rééquilibrage',
             fontsize=14, fontweight='bold', y=1.01)

# AVANT
colors_avant = ['#FF5722' if c == 'No Finding' else '#2196F3' for c in counts_avant.index]
bars0 = axes[0].bar(counts_avant.index, counts_avant.values, color=colors_avant, edgecolor='white', alpha=0.9)
for bar, (lbl, cnt) in zip(bars0, counts_avant.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=7.5, rotation=45)
axes[0].set_title('AVANT rééquilibrage   No Finding domine massivement (rouge)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Nombre images')
axes[0].set_xticks(range(len(counts_avant)))
axes[0].set_xticklabels(counts_avant.index, rotation=45, ha='right', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)
axes[0].axhline(y=counts_avant['No Finding'], color='red', linestyle='--', alpha=0.4, label=f'No Finding = {counts_avant["No Finding"]:,}')
axes[0].legend(fontsize=9)

# APRES
colors_apres = ['#FF9800' if c == 'No Finding' else '#4CAF50' for c in counts_apres.index]
bars1 = axes[1].bar(counts_apres.index, counts_apres.values, color=colors_apres, edgecolor='white', alpha=0.9)
for bar, (lbl, cnt) in zip(bars1, counts_apres.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=7.5, rotation=45)
axes[1].set_title('APRÈS rééquilibrage   Classes équilibrées (orange = No Finding, vert = maladies)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Nombre images')
axes[1].set_xticks(range(len(counts_apres)))
axes[1].set_xticklabels(counts_apres.index, rotation=45, ha='right', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)
axes[1].axhline(y=TARGET_PER_CLASS, color='green', linestyle='--', alpha=0.5, label=f'Cible = {TARGET_PER_CLASS:,}')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/distribution_avant_apres_reequilibrage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé : distribution_avant_apres_reequilibrage.png')

In [ ]:
# =============================================================
# CELLULE A   Split train/val/test AVANT et APRES rééquilibrage
# =============================================================
from sklearn.model_selection import train_test_split

#    Split officiel NIH   
with open(os.path.join(DATA_PATH, 'train_val_list.txt')) as f:
    tv_list = set(f.read().splitlines())
with open(os.path.join(DATA_PATH, 'test_list.txt')) as f:
    te_list = set(f.read().splitlines())

df_trainval = df[df['Image Index'].isin(tv_list)].copy()
df_test     = df[df['Image Index'].isin(te_list)].copy()
df_train, df_val = train_test_split(
    df_trainval, test_size=0.2, random_state=42,
    stratify=df_trainval['No Finding']
)
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

# Sous-ensemble équilibré
sampled = []
for label in ALL_LABELS:
    limit  = 8000 if label == 'No Finding' else 5000
    subset = df_train[
        df_train['Finding Labels'].str.contains(label, na=False)
    ].head(limit)
    sampled.append(subset)
df_train_s = pd.concat(sampled).drop_duplicates(subset='Image Index')
df_train_s = df_train_s.sample(frac=1, random_state=42).reset_index(drop=True)

# Calcul des pourcentages
total_avant = len(df_train) + len(df_val) + len(df_test)
total_apres = len(df_train_s) + len(df_val) + len(df_test)

print('SPLIT DU DATASET   NIH Chest X-ray')
print('='*55)
print(f'{"Split":<15} {"AVANT rééquil.":>18}  {"APRÈS rééquil.":>18}')
print('-'*55)
for name, avant, apres in [
    ('Train',
     f'{len(df_train):,} ({len(df_train)/total_avant*100:.1f}%)',
     f'{len(df_train_s):,} ({len(df_train_s)/total_apres*100:.1f}%)'),
    ('Validation',
     f'{len(df_val):,} ({len(df_val)/total_avant*100:.1f}%)',
     f'{len(df_val):,} ({len(df_val)/total_apres*100:.1f}%)'),
    ('Test',
     f'{len(df_test):,} ({len(df_test)/total_avant*100:.1f}%)',
     f'{len(df_test):,} ({len(df_test)/total_apres*100:.1f}%)'),
    ('TOTAL',
     f'{total_avant:,}',
     f'{total_apres:,}'),
]:
    print(f'{name:<15} {avant:>18}  {apres:>18}')
print(f'\n  Data leakage train/test : '
      f'{len(set(df_train["Image Index"]) & set(df_test["Image Index"]))} (doit être 0)')

# Visualisation avant et apres reequilibrage
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Split du dataset   Avant et Après rééquilibrage',
             fontsize=13, fontweight='bold')

colors = ['#2196F3', '#FF9800', '#4CAF50']

# avant pie chart
sizes_av = [len(df_train), len(df_val), len(df_test)]
lbls_av  = [
    f'Train\n{len(df_train):,}\n({len(df_train)/total_avant*100:.1f}%)',
    f'Validation\n{len(df_val):,}\n({len(df_val)/total_avant*100:.1f}%)',
    f'Test\n{len(df_test):,}\n({len(df_test)/total_avant*100:.1f}%)'
]
axes[0,0].pie(sizes_av, labels=lbls_av, colors=colors,
              autopct='%1.1f%%', startangle=90,
              textprops={'fontsize': 9})
axes[0,0].set_title('AVANT rééquilibrage   Camembert', fontweight='bold')

# apres pie chat
sizes_ap = [len(df_train_s), len(df_val), len(df_test)]
lbls_ap  = [
    f'Train équilibré\n{len(df_train_s):,}\n({len(df_train_s)/total_apres*100:.1f}%)',
    f'Validation\n{len(df_val):,}\n({len(df_val)/total_apres*100:.1f}%)',
    f'Test\n{len(df_test):,}\n({len(df_test)/total_apres*100:.1f}%)'
]
axes[0,1].pie(sizes_ap, labels=lbls_ap, colors=colors,
              autopct='%1.1f%%', startangle=90,
              textprops={'fontsize': 9})
axes[0,1].set_title('APRÈS rééquilibrage   Camembert', fontweight='bold')

# avant barre chart
bars = axes[1,0].bar(['Train', 'Validation', 'Test'],
                     sizes_av, color=colors, edgecolor='white')
for bar, v, p in zip(bars, sizes_av,
                     [len(df_train)/total_avant*100,
                      len(df_val)/total_avant*100,
                      len(df_test)/total_avant*100]):
    axes[1,0].text(bar.get_x()+bar.get_width()/2,
                   bar.get_height()+300,
                   f'{v:,}\n({p:.1f}%)', ha='center', fontsize=9)
axes[1,0].set_ylabel('Nombre images')
axes[1,0].set_title('AVANT rééquilibrage   Barres', fontweight='bold')
axes[1,0].set_ylim([0, max(sizes_av)*1.18])

# apres barre
bars2 = axes[1,1].bar(['Train équilibré', 'Validation', 'Test'],
                      sizes_ap, color=colors, edgecolor='white')
for bar, v, p in zip(bars2, sizes_ap,
                     [len(df_train_s)/total_apres*100,
                      len(df_val)/total_apres*100,
                      len(df_test)/total_apres*100]):
    axes[1,1].text(bar.get_x()+bar.get_width()/2,
                   bar.get_height()+300,
                   f'{v:,}\n({p:.1f}%)', ha='center', fontsize=9)
axes[1,1].set_ylabel('Nombre images')
axes[1,1].set_title('APRÈS rééquilibrage   Barres', fontweight='bold')
axes[1,1].set_ylim([0, max(sizes_av)*1.18])

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/split_avant_apres.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Graphique split sauvegardé : split_avant_apres.png')

In [ ]:
# =============================================================
# CELLULE D   Augmentation uniquement sur les maladies
# Images affichées = pathologies réelles (pas No Finding)
# =============================================================
from tensorflow.keras.preprocessing.image import ImageDataGenerator

df_display = df.reset_index(drop=True)

# exemples pathologies différentes
DEMO_LABELS = [
    'Pneumonia', 'Effusion', 'Atelectasis', 'Cardiomegaly',
    'Emphysema', 'Edema', 'Mass', 'Pneumothorax'
]

# Récuperer image par pathologie
demo_images = {}
for label in DEMO_LABELS:
    samples = df_display[
        df_display['Finding Labels'].str.contains(label, na=False) &
        ~df_display['Finding Labels'].str.contains('No Finding', na=False)
    ]
    if len(samples) > 0:
        demo_images[label] = samples.iloc[0]['FilePath']

print(f'Images trouvées : {len(demo_images)} pathologies')

# Generateur avec toutes les augmentations
aug_gen = ImageDataGenerator(
    rescale           = 1./255,
    rotation_range    = 12,
    horizontal_flip   = True,
    zoom_range        = 0.12,
    width_shift_range = 0.06,
    height_shift_range= 0.06,
    shear_range       = 0.05,
    brightness_range  = [0.85, 1.15],
    fill_mode         = 'nearest'
)
orig_gen = ImageDataGenerator(rescale=1./255)

# Original et augmentation
n_labels = min(len(demo_images), 8)
fig, axes = plt.subplots(n_labels, 2, figsize=(10, n_labels * 2.8))
fig.suptitle(
    'Avant / Après augmentation   Pathologies malades uniquement\n'
    '(No Finding EXCLU de l\'augmentation)',
    fontsize=13, fontweight='bold', y=1.01
)

for row, (label, img_path) in enumerate(list(demo_images.items())[:n_labels]):
    img     = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img     = cv2.resize(img, (224, 224))
    tensor  = np.expand_dims(img, 0)

    # Original
    orig    = next(orig_gen.flow(tensor, batch_size=1, seed=42))[0]
    # Augmentation
    augmented = next(aug_gen.flow(tensor, batch_size=1, seed=row*13))[0]

    axes[row, 0].imshow(orig)
    axes[row, 0].set_title(f'{label}   Original', fontsize=9, color='#1565C0')
    axes[row, 0].axis('off')
    for spine in axes[row, 0].spines.values():
        spine.set_visible(True); spine.set_color('#2196F3'); spine.set_linewidth(2)

    axes[row, 1].imshow(augmented)
    axes[row, 1].set_title(f'{label}   Après augmentation', fontsize=9, color='#2E7D32')
    axes[row, 1].axis('off')
    for spine in axes[row, 1].spines.values():
        spine.set_visible(True); spine.set_color('#4CAF50'); spine.set_linewidth(2)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/augmentation_avant_apres.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Séparer No Finding des autres maladies
df_no_finding = df_train_s[df_train_s['No Finding'] == 1].copy()
df_diseases   = df_train_s[df_train_s['No Finding'] == 0].copy()

print('\nRÉPARTITION DANS LE TRAIN APRÈS RÉÉQUILIBRAGE')
print('='*55)
print(f'  No Finding (sans augmentation) : {len(df_no_finding):,}')
print(f'  Maladies   (avec augmentation) : {len(df_diseases):,}')
print(f'  Total train équilibré          : {len(df_train_s):,}')
N_EPOCHS = 40
print(f'\n  Images malades vues sur {N_EPOCHS} epochs : ~{len(df_diseases)*N_EPOCHS:,}')
print(f'  Transformations appliquées : rotation, flip, zoom,')
print(f'     décalage, shear, variation luminosité')
print(f'  No Finding : aucune augmentation (évite d\'aggraver le déséquilibre)')

In [ ]:
# =============================================================
# CELLULE 4   Augmentation seulement sur les classes malades
# Visualisation avant et apres augmentation
# =============================================================
from tensorflow.keras.preprocessing.ima

# Séparer No Finding des maladies dans le train équilibre
df_no_finding  = df_train_s[df_train_s['No Finding'] == 1].copy()
df_diseases    = df_train_s[df_train_s['No Finding'] == 0].copy()

print(f'No Finding (sans augmentation) : {len(df_no_finding):,}')
print(f'Maladies   (avec augmentation) : {len(df_diseases):,}')

# plusieurs augmentations
disease_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=12,
    horizontal_flip=True,
    zoom_range=0.12,
    width_shift_range=0.06,
    height_shift_range=0.06,
    shear_range=0.05,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

no_finding_datagen = ImageDataGenerator(
    rescale=1./255
)

# Générateur Val/Test
val_test_datagen = ImageDataGenerator(rescale=1./255)


# Visualisation
df_display = df.reset_index(drop=True)
demo_samples = df_display[df_display['Finding Labels'].str.contains('Pneumonia', na=False)]
if len(demo_samples) == 0:
    demo_samples = df_display[df_display['Finding Labels'].str.contains('Effusion', na=False)]
demo_path = demo_samples.iloc[0]['FilePath']

img_orig = cv2.cvtColor(cv2.imread(demo_path), cv2.COLOR_BGR2RGB)
img_orig = cv2.resize(img_orig, (224, 224))

augmentations = [
    ('Original\n(No Finding   inchangé)', ImageDataGenerator(rescale=1./255)),
    ('Rotation ±12°',     ImageDataGenerator(rescale=1./255, rotation_range=12)),
    ('Flip horizontal',   ImageDataGenerator(rescale=1./255, horizontal_flip=True)),
    ('Zoom ±12%',         ImageDataGenerator(rescale=1./255, zoom_range=0.12)),
    ('Décalage horiz.',   ImageDataGenerator(rescale=1./255, width_shift_range=0.06)),
    ('Luminosité ±15%',   ImageDataGenerator(rescale=1./255, brightness_range=[0.85, 1.15])),
    ('Shear 5%',          ImageDataGenerator(rescale=1./255, shear_range=0.05)),
    ('Toutes combinées\n(maladies uniquement)',
     ImageDataGenerator(rescale=1./255, rotation_range=12, horizontal_flip=True,
                        zoom_range=0.12, width_shift_range=0.06,
                        height_shift_range=0.06, shear_range=0.05,
                        brightness_range=[0.85, 1.15], fill_mode='nearest')),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Avant / Après augmentation   Appliquée UNIQUEMENT sur les classes malades\n'
             '(No Finding reste inchangé pour ne pas aggraver le déséquilibre)',
             fontsize=12, fontweight='bold')

img_tensor = np.expand_dims(img_orig, 0)
for i, (ax, (name, gen)) in enumerate(zip(axes.flatten(), augmentations)):
    it  = gen.flow(img_tensor, batch_size=1, seed=i*7)
    aug = next(it)[0]
    ax.imshow(aug, cmap='gray' if aug.shape[-1] == 1 else None)
    color = '#FF5722' if i == 0 else '#4CAF50'
    ax.set_title(name, fontsize=9, fontweight='bold', color=color)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(color)
        spine.set_linewidth(3)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/augmentation_avant_apres.png', dpi=150, bbox_inches='tight')
plt.show()

# calcul nbre avant/après augmentation
print('\nNOMBRE D\'IMAGES   AVANT ET APRÈS AUGMENTATION')
print('='*60)
print(f'  Train AVANT (brut)         : {len(df_train):,}')
print(f'  Train APRÈS rééquilibrage  : {len(df_train_s):,}')
print(f'    dont No Finding          : {len(df_no_finding):,}  (pas augmenté)')
print(f'    dont Maladies            : {len(df_diseases):,}  (augmentées)')
print(f'  Val                        : {len(df_val):,}')
print(f'  Test                       : {len(df_test):,}')
N_EPOCHS_TOTAL = 40
imgs_vues_maladies = len(df_diseases) * N_EPOCHS_TOTAL
print(f'\n  Images malades vues après {N_EPOCHS_TOTAL} epochs : ~{imgs_vues_maladies:,} (avec augmentation à la volée)')
print(f'  Transformations : rotation, flip, zoom, décalage, luminosité, shear')
print('Augmentation ciblée sauvegardée : augmentation_avant_apres.png')

In [ ]:
# =============================================================
# CELLULE 5   FOCAL LOSS + LABEL SMOOTHING
# =============================================================
import tensorflow as tf

def focal_loss(alpha=0.25, gamma=2.0, label_smoothing=0.05):
    """
    Focal Loss avec label smoothing pour classification multi-label.
    - alpha  : poids classe positive
    - gamma  : facteur de focus (0 = crossentropy standard)
    - label_smoothing : évite l'overconfidence
    """
    def loss(y_true, y_pred):
        y_true = y_true * (1.0 - label_smoothing) + 0.5 * label_smoothing
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        
        # Binary cross-entropy de base
        bce = - (y_true * tf.math.log(y_pred) +
                 (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        focal_factor = tf.pow(1.0 - p_t, gamma)
        
        # poids alpha
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        
        focal = alpha_t * focal_factor * bce
        return tf.reduce_mean(focal)
    return loss

# Class weights
class_weights = {}
n = len(df_train_s)
for i, label in enumerate(ALL_LABELS):
    n_pos = df_train_s[label].sum()
    if n_pos > 0:
        w = n / (N_CLASSES * n_pos)
        class_weights[i] = min(w, 10.0)
    else:
        class_weights[i] = 1.0

print('CLASS WEIGHTS APRÈS RÉÉQUILIBRAGE')
print('='*45)
for i, label in enumerate(ALL_LABELS):
    bar = '█' * int(class_weights[i] * 3)
    print(f'  {label:<25} : {class_weights[i]:.3f}  {bar}')

print('\n Focal Loss définie (alpha=0.25, gamma=2.0, label_smoothing=0.05)')
print(' Class weights calculés sur dataset équilibré')

In [ ]:
# =============================================================
# CELLULE 6   DataGenerators
# =============================================================
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# configurations d'augmentation et normalisation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    horizontal_flip=True,
    zoom_range=0.10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Creer des flux d'images
train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train_s,   
    x_col='FilePath', 
    y_col=ALL_LABELS,
    target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE,
    class_mode='raw', 
    color_mode='rgb', 
    shuffle=True, 
    seed=42
)

# validation
val_gen = val_test_datagen.flow_from_dataframe(
    dataframe=df_val, 
    x_col='FilePath', 
    y_col=ALL_LABELS,
    target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE,
    class_mode='raw', 
    color_mode='rgb', 
    shuffle=False, 
    seed=42
)

# test
test_gen = val_test_datagen.flow_from_dataframe(
    dataframe=df_test, 
    x_col='FilePath', 
    y_col=ALL_LABELS,
    target_size=(IMG_SIZE, IMG_SIZE), 
    batch_size=BATCH_SIZE,
    class_mode='raw', 
    color_mode='rgb', 
    shuffle=False, 
    seed=42
)

# calcul automatique des steps pour Keras
steps_per_epoch = len(train_gen)
validation_steps = len(val_gen)

# test de validation du 1er batch
imgs, labs = next(train_gen)

print("\n" + "="*50)
print(f"Générateurs initialisés avec succès !")
print(f"Batch de test : Images {imgs.shape} | Labels {labs.shape}")
print(f"Train (équilibré) : {len(df_train_s)} images ({steps_per_epoch} steps/epoch)")
print(f"Validation      : {len(df_val)} images ({validation_steps} steps)")
print(f"Test            : {len(df_test)} images ({len(test_gen)} steps)")
print("="*50)

In [ ]:
# =============================================================
# CELLULE 7   Architecture DenseNet121 + CBAM + Multi-Scale
# =============================================================
from tensorflow.keras import layers, Model, backend as K
from tensorflow.keras.applications import DenseNet121
import tensorflow as tf

class ChannelMeanPool(layers.Layer):
    def call(self, x):
        return tf.reduce_mean(x, axis=-1, keepdims=True)
    def get_config(self):
        return super().get_config()

class ChannelMaxPool(layers.Layer):
    def call(self, x):
        return tf.reduce_max(x, axis=-1, keepdims=True)
    def get_config(self):
        return super().get_config()

def channel_attention(x, ratio=8):
    channels = x.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D()(x)
    max_pool = layers.GlobalMaxPooling2D()(x)
    shared1  = layers.Dense(channels // ratio, activation='relu')
    shared2  = layers.Dense(channels)
    avg_out  = shared2(shared1(avg_pool))
    max_out  = shared2(shared1(max_pool))
    scale    = layers.Activation('sigmoid')(layers.Add()([avg_out, max_out]))
    scale    = layers.Reshape((1, 1, channels))(scale)
    return layers.Multiply()([x, scale])

def spatial_attention(x):
    avg_pool = ChannelMeanPool()(x)
    max_pool = ChannelMaxPool()(x)
    concat   = layers.Concatenate()([avg_pool, max_pool])
    scale    = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)
    return layers.Multiply()([x, scale])

def cbam_block(x):
    x = channel_attention(x)
    x = spatial_attention(x)
    return x

from tensorflow.keras.regularizers import l2

def build_model(n_classes=15):
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    base   = DenseNet121(weights='imagenet', include_top=False, input_tensor=inputs)

    # PHASE 1 : tout le backbone gelé
    for layer in base.layers:
        layer.trainable = False

    feat_small  = base.get_layer('conv4_block6_concat').output
    feat_medium = base.get_layer('conv3_block12_concat').output
    feat_large  = base.output

    att_large  = cbam_block(feat_large)
    att_medium = cbam_block(feat_medium)
    att_small  = cbam_block(feat_small)

    p_large  = layers.GlobalAveragePooling2D()(att_large)
    p_medium = layers.GlobalAveragePooling2D()(att_medium)
    p_small  = layers.GlobalAveragePooling2D()(att_small)

    fused = layers.Concatenate()([p_large, p_medium, p_small])

    # L2 regularization
    x = layers.Dense(1024, kernel_regularizer=l2(1e-4))(fused)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)   # 0.4 → 0.5

    shortcut = layers.Dense(512, kernel_regularizer=l2(1e-4))(fused)
    x2       = layers.Dense(512, kernel_regularizer=l2(1e-4))(x)
    x2       = layers.BatchNormalization()(x2)
    x2       = layers.Activation('relu')(x2)
    x2       = layers.Dropout(0.4)(x2)  # 0.3 → 0.4
    x2       = layers.Add()([x2, shortcut])

    outputs = layers.Dense(n_classes, activation='sigmoid')(x2)
    return Model(inputs=inputs, outputs=outputs)

model = build_model(N_CLASSES)
print(f'Paramètres totaux   : {model.count_params():,}')
print(f'Paramètres entraîn. : {sum(K.count_params(w) for w in model.trainable_weights):,}')

In [ ]:
# =============================================================
# CELLULE 8   Compilation avec Focal Loss
# =============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=focal_loss(alpha=0.25, gamma=1.5, label_smoothing=0.05),
    metrics=[
        AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

callbacks = [
    ModelCheckpoint(
        f'{WORK_PATH}/models/best_model.keras',
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_auc', patience=8, mode='max',
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_auc', factor=0.3, patience=4,
        min_lr=1e-6, mode='max', verbose=1
    ),
    CSVLogger(f'{WORK_PATH}/logs/training_log.csv'),
]

print('Modèle compilé avec Focal Loss')
print('Métriques : AUC (multi-label), Precision, Recall')
print('Callbacks : ModelCheckpoint, EarlyStopping (patience=8), ReduceLROnPlateau')

In [ ]:
# PHASE 1 : tete seulement, backbone gelé
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=focal_loss(alpha=0.25, gamma=1.5, label_smoothing=0.05),
    metrics=[
        AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

callbacks = [
    ModelCheckpoint(
        f'{WORK_PATH}/models/best_model.keras',
        monitor='val_auc', mode='max', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_auc', patience=8, mode='max',
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_auc', factor=0.3, patience=4,
        min_lr=1e-6, mode='max', verbose=1
    ),
    CSVLogger(f'{WORK_PATH}/logs/training_log.csv'),
]

print('PHASE 1   Backbone gelé, LR=1e-4, 15 epochs')
history1 = model.fit(
    train_gen,
    steps_per_epoch=len(train_gen),
    epochs=15,
    validation_data=val_gen,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

# PHASE 2 : fine-tuning, 30 dernières couches seulement
base_model = model.layers[1]  # DenseNet121
for layer in base_model.layers:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=focal_loss(alpha=0.25, gamma=1.5, label_smoothing=0.05),
    metrics=[
        AUC(name='auc', multi_label=True),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

print('PHASE 2   Fine-tuning 30 couches, LR=5e-5, 25 epochs')
history2 = model.fit(
    train_gen,
    steps_per_epoch=len(train_gen),
    epochs=25,
    validation_data=val_gen,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

In [ ]:
# for i, layer in enumerate(model.layers):
#     print(f'{i:3d} | {type(layer).__name__:<25} | {layer.name}')

In [ ]:
# # =============================================================
# # PHASE 2   Fine-tuning
# # =============================================================

# for layer in model.layers[:427]:
#     layer.trainable = False

# # Degele 30 dernières couches DenseNet
# for layer in model.layers[397:427]:
#     layer.trainable = True

# # verification
# total_trainable = sum(1 for l in model.layers if l.trainable)
# densenet_unfreeze = sum(1 for l in model.layers[397:427] if l.trainable)
# heads_trainable   = sum(1 for l in model.layers[427:] if l.trainable)

# print(f'Couches DenseNet dégelées : {densenet_unfreeze}')
# print(f'  De : {model.layers[397].name}')
# print(f'  À  : {model.layers[426].name}')
# print(f'Couches tête (CBAM+Dense) : {heads_trainable}')
# print(f'Total entraînable         : {total_trainable}')

# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
#     loss=focal_loss(alpha=0.25, gamma=1.5, label_smoothing=0.05),
#     metrics=[
#         AUC(name='auc', multi_label=True),
#         tf.keras.metrics.Precision(name='precision'),
#         tf.keras.metrics.Recall(name='recall'),
#     ]
# )

# callbacks_p2 = [
#     ModelCheckpoint(
#         f'{WORK_PATH}/models/best_model_phase2.keras',
#         monitor='val_auc', mode='max', save_best_only=True, verbose=1
#     ),
#     EarlyStopping(
#         monitor='val_auc', patience=8, mode='max',
#         restore_best_weights=True, verbose=1
#     ),
#     ReduceLROnPlateau(
#         monitor='val_auc', factor=0.3, patience=4,
#         min_lr=1e-7, mode='max', verbose=1
#     ),
#     CSVLogger(f'{WORK_PATH}/logs/training_log_phase2.csv'),
# ]

# print('\nPHASE 2   Fine-tuning conv5 final, LR=5e-5, 25 epochs')
# history2 = model.fit(
#     train_gen,
#     steps_per_epoch=len(train_gen),
#     epochs=13,
#     validation_data=val_gen,
#     callbacks=callbacks_p2,
#     class_weight=class_weights,
#     verbose=1
# )

In [ ]:
# =============================================================
# CELLULE 10   Courbes d'apprentissage
# =============================================================

combined_history = {}
for key in history1.history:
    val_key = f'val_{key}' if f'val_{key}' in history1.history else None
    combined_history[key] = history1.history[key] + history2.history.get(key, [])
for key in list(combined_history.keys()):
    vk = f'val_{key}'
    if vk in history1.history:
        combined_history[vk] = history1.history[vk] + history2.history.get(vk, [])

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Courbes d\'apprentissage   DenseNet121 + CBAM + Focal Loss', fontsize=13, fontweight='bold')

metrics_to_plot = [('loss', 'Focal Loss'), ('auc', 'AUC-ROC'),
                   ('precision', 'Precision'), ('recall', 'Recall')]

for ax, (metric, title) in zip(axes.flatten(), metrics_to_plot):
    ax.plot(combined_history[metric], label='Train', color='#2196F3', linewidth=2)
    ax.plot(combined_history[f'val_{metric}'], label='Val', color='#FF5722', linewidth=2)
    ax.axvline(x=len(history1.history[metric])-1, color='gray', linestyle='--', alpha=0.6, label='Fine-tuning →')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/learning_curves.png', dpi=150)
plt.show()
print('Courbes sauvegardées : learning_curves.png')

In [ ]:
# =============================================================
# CELLULE 11   euils optimaux par classe
# =============================================================
from sklearn.metrics import precision_recall_curve, f1_score

def make_gen(datagen, df_data, shuffle):
    return datagen.flow_from_dataframe(
        dataframe=df_data, x_col='FilePath', y_col=ALL_LABELS,
        target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode='raw', color_mode='rgb', shuffle=shuffle, seed=42
    )

from tensorflow.keras.preprocessing.image import ImageDataGenerator
val_test_datagen = ImageDataGenerator(rescale=1./255)

print('Calcul des prédictions sur le set de validation...')
y_val_true  = np.zeros((len(df_val), N_CLASSES))
y_val_proba = np.zeros((len(df_val), N_CLASSES))

val_gen_fresh = make_gen(val_test_datagen, df_val, shuffle=False)
for i in range(len(val_gen_fresh)):
    x_batch, y_batch = next(val_gen_fresh)
    preds = model.predict(x_batch, verbose=0)
    start = i * BATCH_SIZE
    end   = min(start + len(x_batch), len(df_val))
    y_val_true[start:end]  = y_batch[:end-start]
    y_val_proba[start:end] = preds[:end-start]

# seuil optimal par classe
optimal_thresholds = {}

print('\nSEUILS OPTIMAUX PAR CLASSE (maximisation F1 sur val)')
print('='*60)
print(f'{"Pathologie":<25} {"Seuil opt":>10}  {"F1@0.5":>8}  {"F1@optimal":>12}')
print('-'*60)

for i, label in enumerate(ALL_LABELS):
    if y_val_true[:, i].sum() == 0:
        optimal_thresholds[label] = 0.5
        continue

    precision_arr, recall_arr, thresholds = precision_recall_curve(
        y_val_true[:, i], y_val_proba[:, i]
    )

    # F1 
    f1_arr = 2 * precision_arr[:-1] * recall_arr[:-1] / (
        precision_arr[:-1] + recall_arr[:-1] + 1e-8
    )

    best_idx       = np.argmax(f1_arr)
    best_threshold = thresholds[best_idx]
    optimal_thresholds[label] = float(best_threshold)

    # Comparaison
    f1_fixed   = f1_score(y_val_true[:, i], (y_val_proba[:, i] >= 0.5).astype(int), zero_division=0)
    f1_optimal = f1_score(y_val_true[:, i], (y_val_proba[:, i] >= best_threshold).astype(int), zero_division=0)
    gain = 'done' if f1_optimal > f1_fixed else ''
    print(f'{label:<25} {best_threshold:>10.3f}  {f1_fixed:>8.3f}  {f1_optimal:>12.3f}  {gain}')

print('-'*60)

# sauvegarde des seuils
with open(f'{WORK_PATH}/models/optimal_thresholds.json', 'w') as f:
    json.dump(optimal_thresholds, f, indent=2)

print(f'\n Seuils sauvegardés : optimal_thresholds.json')
print('Avantage : recall amélioré sur les classes rares (Hernia, Pneumonia...)')

In [ ]:
# =============================================================
# CELLULE 12   Évaluation finale sur le TEST SET
# =============================================================
from sklearn.metrics import accuracy_score

print('Prédictions sur le TEST SET...')
y_true  = np.zeros((len(df_test), N_CLASSES))
y_pred_proba = np.zeros((len(df_test), N_CLASSES))

test_gen_fresh = make_gen(val_test_datagen, df_test, shuffle=False)
for i in range(len(test_gen_fresh)):
    x_batch, y_batch = next(test_gen_fresh)
    preds = model.predict(x_batch, verbose=0)
    start = i * BATCH_SIZE
    end   = min(start + len(x_batch), len(df_test))
    y_true[start:end]       = y_batch[:end-start]
    y_pred_proba[start:end] = preds[:end-start]

# applications sur les seuils optimaux
y_pred_bin = np.zeros_like(y_pred_proba)
for i, label in enumerate(ALL_LABELS):
    threshold = optimal_thresholds.get(label, 0.5)
    y_pred_bin[:, i] = (y_pred_proba[:, i] >= threshold).astype(int)

# metriques
auc_per_class = []
for i, label in enumerate(ALL_LABELS):
    if y_true[:, i].sum() > 0:
        val = roc_auc_score(y_true[:, i], y_pred_proba[:, i])
        auc_per_class.append(val)
        bar = '█' * int(val * 25)
        print(f'  {label:<25} : {val:.4f}  {bar}')
    else:
        auc_per_class.append(None)
        print(f'  {label:<25} : N/A')

valid_aucs = [a for a in auc_per_class if a is not None]
mean_auc   = np.mean(valid_aucs)

f1_macro    = f1_score(y_true, y_pred_bin, average='macro',    zero_division=0)
f1_micro    = f1_score(y_true, y_pred_bin, average='micro',    zero_division=0)
f1_weighted = f1_score(y_true, y_pred_bin, average='weighted', zero_division=0)
prec_macro  = precision_score(y_true, y_pred_bin, average='macro',    zero_division=0)
rec_macro   = recall_score(y_true, y_pred_bin, average='macro',       zero_division=0)
acc_global  = accuracy_score(y_true.flatten(), y_pred_bin.flatten())

print(f'\nMÉTRIQUES GLOBALES   TEST SET (seuils optimaux par classe)')
print('='*55)
print(f'  Mean AUC-ROC       : {mean_auc:.4f}  ← indicateur principal')
print(f'  Accuracy globale   : {acc_global:.4f}  ({acc_global*100:.2f}%)')
print(f'  F1-score macro     : {f1_macro:.4f}')
print(f'  F1-score micro     : {f1_micro:.4f}')
print(f'  F1-score weighted  : {f1_weighted:.4f}')
print(f'  Precision macro    : {prec_macro:.4f}')
print(f'  Recall macro       : {rec_macro:.4f}')

print(f'\nTABLEAU PAR PATHOLOGIE (seuils optimaux)')
print(f'{"Pathologie":<25} {"Seuil":>6} {"AUC":>6}  {"F1":>6}  {"Precision":>9}  {"Recall":>8}')
print('-'*75)
for i, label in enumerate(ALL_LABELS):
    if y_true[:, i].sum() == 0:
        continue
    thr   = optimal_thresholds.get(label, 0.5)
    auc_v = auc_per_class[i] if auc_per_class[i] else 0
    f1_v  = f1_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    pre_v = precision_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    rec_v = recall_score(y_true[:,i], y_pred_bin[:,i], zero_division=0)
    print(f'{label:<25} {thr:>6.3f} {auc_v:>6.3f}  {f1_v:>6.3f}  {pre_v:>9.3f}  {rec_v:>8.3f}')
print('-'*75)
print(f'{"MOYENNE":<25}       {mean_auc:>6.3f}  {f1_macro:>6.3f}  {prec_macro:>9.3f}  {rec_macro:>8.3f}')

print(f'\n{classification_report(y_true, y_pred_bin, target_names=ALL_LABELS, zero_division=0)}')

In [ ]:
# =============================================================
# CELLULE 13   Visualisations complètes des métriques
# =============================================================

valid_labels = [ALL_LABELS[i] for i, a in enumerate(auc_per_class) if a is not None]
valid_vals   = [a for a in auc_per_class if a is not None]
colors_bar   = ['#4CAF50' if v >= 0.80 else '#FF9800' if v >= 0.70 else '#F44336'
                for v in valid_vals]

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Métriques complètes   DenseNet121 + CBAM + Focal Loss + Seuils Optimaux',
             fontsize=13, fontweight='bold')

# AUC par classe
bars = axes[0, 0].bar(valid_labels, valid_vals, color=colors_bar, edgecolor='white')
axes[0, 0].axhline(y=mean_auc, color='navy', linestyle='--', linewidth=2, label=f'Mean AUC = {mean_auc:.4f}')
axes[0, 0].axhline(y=0.80, color='green', linestyle=':', label='Objectif 0.80')
axes[0, 0].set_ylim([0.5, 1.0])
axes[0, 0].set_title('AUC-ROC par pathologie', fontweight='bold')
axes[0, 0].set_ylabel('AUC-ROC')
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=8)
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, valid_vals):
    axes[0, 0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=7)

# F1 / Precision / Recall
f1_vals   = [f1_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]
prec_vals = [precision_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]
rec_vals  = [recall_score(y_true[:,i], y_pred_bin[:,i], zero_division=0) for i in range(N_CLASSES)]

x = np.arange(N_CLASSES)
w = 0.28
axes[0, 1].bar(x-w, f1_vals,   width=w, label='F1-score',  color='#2196F3', alpha=0.85)
axes[0, 1].bar(x,   prec_vals, width=w, label='Precision', color='#4CAF50', alpha=0.85)
axes[0, 1].bar(x+w, rec_vals,  width=w, label='Recall',    color='#FF9800', alpha=0.85)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(ALL_LABELS, rotation=45, ha='right', fontsize=8)
axes[0, 1].set_title('F1 / Precision / Recall par pathologie', fontweight='bold')
axes[0, 1].set_ylim([0, 1.0])
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)

# Courbes ROC Top 5
top5    = sorted(range(len(ALL_LABELS)),
                 key=lambda i: auc_per_class[i] if auc_per_class[i] else 0, reverse=True)[:5]
palette = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
for idx, color in zip(top5, palette):
    fpr, tpr, _ = roc_curve(y_true[:, idx], y_pred_proba[:, idx])
    axes[1, 0].plot(fpr, tpr, color=color, linewidth=2,
                    label=f'{ALL_LABELS[idx]} (AUC={auc_per_class[idx]:.3f})')
axes[1, 0].plot([0,1],[0,1],'k--', linewidth=1, label='Aléatoire')
axes[1, 0].set_xlabel('Taux faux positifs')
axes[1, 0].set_ylabel('Taux vrais positifs')
axes[1, 0].set_title('Courbes ROC   Top 5 pathologies', fontweight='bold')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# resume metriques
metric_names = ['Mean AUC', 'Accuracy', 'F1 Macro', 'F1 Micro', 'F1 Weighted', 'Precision', 'Recall']
metric_vals  = [mean_auc, acc_global, f1_macro, f1_micro, f1_weighted, prec_macro, rec_macro]
metric_colors = ['#4CAF50' if v >= 0.75 else '#FF9800' if v >= 0.60 else '#F44336' for v in metric_vals]
bars2 = axes[1, 1].barh(metric_names, metric_vals, color=metric_colors, edgecolor='white')
for bar, val in zip(bars2, metric_vals):
    axes[1, 1].text(val + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
axes[1, 1].axvline(x=0.75, color='green', linestyle='--', alpha=0.6, label='Objectif 0.75')
axes[1, 1].set_xlim([0, 1.1])
axes[1, 1].set_title('Métriques globales   Résumé', fontweight='bold')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{WORK_PATH}/logs/metriques_completes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Métriques complètes sauvegardées : metriques_completes.png')

In [ ]:
# =============================================================
# CELLULE 14   Sauvegarde finale
# =============================================================

# sauvegarde du modele
model.save(f'{WORK_PATH}/models/model_final.keras')

with open(f'{WORK_PATH}/models/labels.json', 'w') as f:
    json.dump(ALL_LABELS, f)

metrics_export = {
    'mean_auc'          : float(mean_auc),
    'f1_macro'          : float(f1_macro),
    'f1_micro'          : float(f1_micro),
    'f1_weighted'       : float(f1_weighted),
    'accuracy'          : float(acc_global),
    'precision_macro'   : float(prec_macro),
    'recall_macro'      : float(rec_macro),
    'img_size'          : IMG_SIZE,
    'n_classes'         : N_CLASSES,
    'labels'            : ALL_LABELS,
    'model_name'        : 'DenseNet121_CBAM_MultiScale_FocalLoss',
    'architecture'      : 'DenseNet121 + CBAM + Multi-Scale + Focal Loss + Seuils Optimaux',
    'corrections'       : [
        'Undersampling No Finding au niveau des classes maladies',
        'Augmentation ciblée : maladies uniquement (pas No Finding)',
        'Focal Loss (alpha=0.25, gamma=1.5) + Label Smoothing (0.05)',
        'Seuils optimaux par classe (maximisation F1 sur val)',
        'Dropout 0.5/0.4 + L2 regularization (1e-4)',
        'Class weights recalculés sur dataset équilibré',
        'Fine-tuning 2 phases : backbone gelé puis 30 dernières couches',
    ],
    'training'          : {
        'phase1_epochs' : len(history1.history['val_auc']),
        'phase2_epochs' : len(history2.history['val_auc']),
        'best_val_auc_p1' : float(max(history1.history['val_auc'])),
        'best_val_auc_p2' : float(max(history2.history['val_auc'])),
        'lr_phase1'     : 1e-4,
        'lr_phase2'     : 5e-5,
    },
    'optimal_thresholds': optimal_thresholds,
    'auc_per_class'     : {
        label: float(auc_per_class[i])
        for i, label in enumerate(ALL_LABELS) if auc_per_class[i] is not None
    }
}

with open(f'{WORK_PATH}/models/metrics.json', 'w') as f:
    json.dump(metrics_export, f, indent=2)

import os
files_to_check = [
    f'{WORK_PATH}/models/model_final.keras',
    f'{WORK_PATH}/models/best_model_phase2.keras',
    f'{WORK_PATH}/models/labels.json',
    f'{WORK_PATH}/models/metrics.json',
    f'{WORK_PATH}/models/optimal_thresholds.json',
]
print('FICHIERS SAUVEGARDÉS')
print('='*50)
for f_path in files_to_check:
    exists = 'done' if os.path.exists(f_path) else 'not found'
    size   = f'{os.path.getsize(f_path)/1024:.1f} KB' if os.path.exists(f_path) else ''
    print(f'  {exists}  {os.path.basename(f_path):<40} {size}')

print(f'\nRÉSUMÉ FINAL   Phase 1 + Phase 2')
print('='*50)
print(f'  Phase 1 epochs  : {len(history1.history["val_auc"])}')
print(f'  Phase 2 epochs  : {len(history2.history["val_auc"])}')
print(f'  Best AUC Phase1 : {max(history1.history["val_auc"]):.4f}')
print(f'  Best AUC Phase2 : {max(history2.history["val_auc"]):.4f}')
print(f'\nMÉTRIQUES TEST SET (seuils optimaux par classe)')
print('='*50)
print(f'  Mean AUC    : {mean_auc:.4f}')
print(f'  Accuracy    : {acc_global:.4f} ({acc_global*100:.2f}%)')
print(f'  F1 macro    : {f1_macro:.4f}')
print(f'  Precision   : {prec_macro:.4f}')
print(f'  Recall      : {rec_macro:.4f}')